In [1]:
import pandas as pd
import sklearn
import ipykernel
#print(f"Ipykernel version: {ipykernel.__version__}")
#print("Environment is ready for data analysis and machine learning tasks.")

In [2]:
## Data Preparation
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load dataset
data = pd.read_csv("dataset.csv")

#print("Original Dataset:")
#print(data.head())

# Remove missing values
data = data.dropna()

# Standardize column names
data.columns = data.columns.str.lower()

# Select important music features
features = ["tempo", "energy", "valence", "instrumentalness", "danceability"]
music_features = data[features]

# Normalize the data
scaler = MinMaxScaler()
normalized_data = scaler.fit_transform(music_features)

# Convert normalized data back to dataframe
normalized_df = pd.DataFrame(normalized_data, columns=features)

# Combine song information with normalized features
clean_data = pd.concat([data[["track_name", "artists"]].reset_index(drop=True), normalized_df], axis=1)

#print("\nCleaned Dataset:")
#print(clean_data.head())

# Save cleaned dataset
clean_data.to_csv("clean_music_dataset.csv", index=False)


#print("\nData preparation completed successfully.")

In [3]:
## Hybrid AI Similarity Engine 
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

#1. Load the cleaned data 
try:
    music_df = pd.read_csv("clean_music_dataset.csv")
except FileNotFoundError:
    print("Error: clean_music_dataset.csv not found. Please run Data Preparation cell first.")

#2. Define the Target Vectors from Table 4.4.1
#Format: [tempo, energy, valence, instrumentalness, danceability]
study_profiles = {
    "Deep Study": [0.35, 0.30, 0.40, 0.85, 0.35],
    "Creative Work": [0.55, 0.60, 0.65, 0.40, 0.55],
    "Relaxation": [0.30, 0.25, 0.70, 0.50, 0.30],
    "Active Learning": [0.65, 0.70, 0.60, 0.30, 0.65]
}

def get_recommendations(situation):
    """
    AI Engine: Calculates cosine similarity between a 
    chosen study situation and the music dataset.
    """
    music_df.dropna(inplace=True)
    
    #Define the features to compare
    features = ["tempo", "energy", "valence", "instrumentalness", "danceability"]
    
    #Get the specific target vector for the selected situation
    target_vector = [study_profiles[situation]]
    
    #Calculate similarity scores against the entire dataset
    #Inference Pipeline logic
    scores = cosine_similarity(target_vector, music_df[features])
    
    #Add scores to a copy of the dataframe to keep original data safe
    results_df = music_df.copy()
    results_df['similarity'] = scores[0]
    
    #Sort by highest similarity and take Top 10 to reduce decision fatigue
    top_10 = results_df.sort_values(by='similarity', ascending=False).head(10)
    
    #Return list of tuples (Track Name, Artist) for the UI to display
    return list(zip(top_10['track_name'], top_10['artists']))

#print("AI Engine initialized and ready for integration!")

In [4]:
# Interactive UI Components 
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Injecting Responsive Desktop CSS for styling
display(HTML("""
<style>
    * { box-sizing: border-box !important; }
    .main-app-font { font-family: -apple-system, BlinkMacSystemFont, "SF Pro Display", sans-serif; }
    
    /* Animated mesh gradient background */
    .mesh-gradient {
        background: linear-gradient(45deg, #ff9a9e 0%, #fad0c4 25%, #fad0c4 50%, #a1c4fd 75%, #c2e9fb 100%);
        background-size: 400% 400%;
        animation: gradientAnimation 15s ease infinite;
    }
    @keyframes gradientAnimation { 0% { background-position: 0% 50%; } 50% { background-position: 100% 50%; } 100% { background-position: 0% 50%; } }

    /* Header Title Styling */
    .desktop-header-text h1 { margin: 0; font-size: 32px; color: #111; font-weight: 800; letter-spacing: -0.5px; }
    
    /* UI FIX: Force the situational buttons to stay in a horizontal row */
    .my-custom-toggles, .my-custom-toggles > div {
        display: flex !important;
        flex-direction: row !important; 
        flex-wrap: nowrap !important;   
        justify-content: center !important;
        align-items: center !important;
        width: 100% !important;
    }
    
    /* Styling for the capsule-shaped background of the situation selector */
    .my-custom-toggles {
        background: rgba(0, 0, 0, 0.04) !important; 
        border-radius: 40px !important;
        padding: 6px !important;
        margin-bottom: 30px !important;
        height: auto !important; 
    }

    /* Styling for individual buttons within the situation selector */
    .my-custom-toggles button {
        flex: 1 1 0% !important; 
        height: 45px !important; 
        min-width: 120px !important;
        white-space: nowrap !important;
        background: transparent !important;
        color: #888 !important; 
        border: none !important;
        border-radius: 30px !important;
        font-weight: 600 !important;
        font-size: 15px !important;
        transition: all 0.3s ease !important;
        margin: 0 4px !important; 
        
        /* Center text content within buttons */
        display: flex !important;
        align-items: center !important;
        justify-content: center !important;
        padding: 0 !important; 
    }
    
    /* Active/Pressed state for buttons */
    .my-custom-toggles button[aria-pressed="true"], .my-custom-toggles button.mod-active, .my-custom-toggles button.active {
        background: white !important; color: #ff4d6d !important; box-shadow: 0 4px 15px rgba(0,0,0,0.08) !important;
    }

    /* User Profile Button Styling */
    .profile-btn { background: rgba(0,0,0,0.04) !important; color: #111 !important; border-radius: 30px !important; font-weight: 600 !important; border: none !important; transition: background 0.2s ease !important; font-size: 14px !important; }
    .profile-btn:hover { background: rgba(0,0,0,0.08) !important; }

    /* User Popup Menu Item Styling */
    .menu-item-btn { background: white !important; color: #333 !important; border: none !important; text-align: left !important; padding: 10px 15px !important; width: 100% !important; font-weight: 500 !important; border-radius: 8px !important; }
    .menu-item-btn:hover { background: #f5f5f5 !important; color: #ff4d6d !important; }

    /* "Generate Playlist" Button Styling */
    .generate-btn { background: white !important; border-radius: 30px !important; border: none !important; color: #ff4d6d !important; font-weight: 800 !important; font-size: 16px !important; box-shadow: 0 8px 25px rgba(0,0,0,0.1) !important; transition: all 0.2s ease !important; }
    .generate-btn:hover { transform: scale(1.05) !important; background: #ff4d6d !important; color: white !important; }
</style>
"""))

# --- Create UI Components ---

# 1. App Header Title
header_title = widgets.HTML('<div class="main-app-font desktop-header-text"><h1>Listen Now</h1></div>')

# 2. User Profile Button (Top Right)
profile_button = widgets.Button(description='👤 User Profile', layout=widgets.Layout(width='140px', height='45px'))
profile_button.add_class('profile-btn').add_class('main-app-font')

header_layout = widgets.HBox([header_title, profile_button], 
                             layout=widgets.Layout(width='100%', justify_content='space-between', align_items='center', 
                                                   border_bottom='1px solid rgba(0,0,0,0.06)', padding='0 0 15px 0'))

# 3. Hidden User Settings Menu (Toggled by Profile Button)
btn_settings = widgets.Button(description='⚙️ Settings', layout=widgets.Layout(width='100%', height='40px')).add_class('menu-item-btn').add_class('main-app-font')
btn_logout = widgets.Button(description='🚪 Log Out', layout=widgets.Layout(width='100%', height='40px')).add_class('menu-item-btn').add_class('main-app-font')

user_menu = widgets.VBox([btn_settings, btn_logout], layout=widgets.Layout(
    display='none', 
    width='160px', background_color='white', border='1px solid rgba(0,0,0,0.08)',
    box_shadow='0 15px 35px rgba(0,0,0,0.15)', border_radius='12px', padding='8px'
))

# 4. Situation Selector (The 4 Interactive Situations)
situation_select = widgets.ToggleButtons(
    options=['Deep Study', 'Creative Work', 'Relaxation', 'Active Learning'],
    value='Deep Study', 
    description='',
    style={'button_width': '140px'} 
)
# Force horizontal alignment via Python Layout
situation_select.layout = widgets.Layout(
    display='flex', 
    flex_flow='row nowrap', 
    align_items='center', 
    justify_content='center', 
    width='100%'
)
situation_select.add_class('main-app-font').add_class('my-custom-toggles')

# 5. The Main Action Button
generate_button = widgets.Button(description='Generate Playlist', layout=widgets.Layout(width='250px', height='55px'))
generate_button.add_class('generate-btn').add_class('main-app-font')

# Output containers for dynamic content
playlist_output = widgets.Output()

In [ ]:
import base64

def get_base64_audio(file_path):
    """Encodes the audio file to a string so the browser doesn't need to request it from the server."""
    try:
        with open(file_path, "rb") as f:
            data = f.read()
            return base64.b64encode(data).decode()
    except FileNotFoundError:
        return ""
    
def generate_playlist_ui(b):
    mode = situation_select.value
    tracks = get_recommendations(mode)
    
    # Map each situation to a specific vibe-matching file
    audio_map = {
        'Deep Study': 'deep_study.mp3',
        'Creative Work': 'creative_work.mp3',
        'Relaxation': 'relaxation.mp3',
        'Active Learning': 'active_learning.mp3'
    }

    # Convert the music into a data string so the browser can play it without needing to request it from the server
    file_path = audio_map.get(mode, "relaxation.mp3")
    audio_base64 = get_base64_audio(file_path)
    audio_src = f"data:audio/mp3;base64,{audio_base64}"

    with playlist_output:
        clear_output()
        
        # --- JavaScript for 10-second Audio Preview Logic ---
        js_code = """
        <script>
        function playPreview(audioId, btnId) {
            var audio = document.getElementById(audioId);
            var btn = document.getElementById(btnId);
            
            // Toggle pause if the current track is already playing
            if (!audio.paused) {
                audio.pause();
                audio.currentTime = 0; 
                btn.innerHTML = '▶';
                return;
            }
            
            // Stop all other playing tracks before starting new preview
            var allAudios = document.querySelectorAll('.preview-audio');
            var allBtns = document.querySelectorAll('.play-btn');
            allAudios.forEach(function(a) { a.pause(); a.currentTime = 0; });
            allBtns.forEach(function(b) { b.innerHTML = '▶'; });
            
            // Start playback for selected track
            audio.play();
            btn.innerHTML = '⏸'; 
            
            // CORE REQUIREMENT: Automatically stop playback after 10 seconds
            setTimeout(function() {
                if (!audio.paused) {
                    audio.pause();
                    audio.currentTime = 0;
                    btn.innerHTML = '▶'; 
                }
            }, 10000); 
        }
        </script>
        """
        
        # Main Playlist Result Card (Mesh Gradient)
        html = js_code + f"""
        <div class='main-app-font mesh-gradient' style='
            padding: 50px 60px; border-radius:40px;
            width:100%; max-width:900px; margin:20px auto; 
            box-shadow: 0 30px 60px rgba(0,0,0,0.2); border: 1px solid rgba(255,255,255,0.3);'>
            
            <h2 style='color:white; font-size:46px; font-weight:800; margin-top:0; margin-bottom:35px; text-align:left; letter-spacing:-1.5px;'>
                {mode}
            </h2>
        """
        
        # Generate individual track rows
        for i, (track_name, artist_name) in enumerate(tracks, 1):

            html += f"""
            <div style='
                display:flex; justify-content:space-between; align-items:center;
                padding:25px 30px; margin-bottom:18px; border-radius:28px; 
                background: rgba(255,255,255,0.15); backdrop-filter: blur(20px);
                border: 1px solid rgba(255,255,255,0.2);'>
                <div>
                    <div style='color:white; font-weight:700; font-size:24px;'>{track_name}</div>
                    <div style='color:rgba(255,255,255,0.8); font-size:16px; margin-top:6px;'>{artist_name}</div>
                </div>
                
                <audio id='audio_{i}' class='preview-audio' src='{audio_src}'></audio>
                
                <button id='btn_{i}' class='play-btn' onclick='playPreview("audio_{i}", "btn_{i}")' 
                        style='border:none; border-radius:50%; width:60px; height:60px;
                               background: white; color:#ff4d6d; cursor:pointer;
                               display:flex; align-items:center; justify-content:center;
                               box-shadow: 0 4px 15px rgba(0,0,0,0.1); font-size:24px; padding-left:5px;'>
                    ▶
                </button>
            </div>
            """
        html += "</div>"
        display(HTML(html))

# Function to show/hide the User Profile menu
def toggle_user_menu(b):
    if user_menu.layout.display == 'none':
        user_menu.layout.display = 'flex'
    else:
        user_menu.layout.display = 'none'

# Event Listeners
generate_button.on_click(generate_playlist_ui)
profile_button.on_click(toggle_user_menu)

In [6]:
# Create a right-aligned container for the popup menu (positioned under profile button)
menu_container = widgets.HBox([user_menu], layout=widgets.Layout(width='100%', justify_content='flex-end', margin='10px 0 0 0'))

# Contextual Situation Suggester Logic
suggester_mapping = {
    "exam": "Deep Study", "test": "Deep Study", "final": "Deep Study", "study": "Deep Study",
    "coding": "Creative Work", "design": "Creative Work", "write": "Creative Work",
    "tired": "Relaxation", "stress": "Relaxation", "break": "Relaxation", "rest": "Relaxation",
    "group": "Active Learning", "project": "Active Learning", "discuss": "Active Learning"
}

mood_label = widgets.HTML("<div class='main-app-font' style='color:#666; font-weight:700; margin-bottom:10px; font-size:16px;'>How are you feeling?</div>")

# textbox
mood_input_box = widgets.Text(
    placeholder='How are you feeling? (e.g., "I have an exam")',
    description='Mood Input:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px', margin='0 0 20px 0')
)

def handle_mood_input(change):
    user_text = change['new'].lower()
    for word, mode in suggester_mapping.items():
        if word in user_text:
            situation_select.value = mode 
            break

mood_input_box.observe(handle_mood_input, names='value')

# Layout for the central selection area (Situation text + ToggleButtons + Generate Button)
controls_layout = widgets.VBox(
    [
        widgets.HTML("<div class='main-app-font' style='color:#666; font-weight:700; margin-bottom:15px; font-size:16px;'>Select Your Vibe:</div>"),
        mood_input_box,
        situation_select, 
        generate_button
    ],
    layout=widgets.Layout(align_items='center', margin='40px 0 40px 0', width='100%')
)

# Final Main Application Container (Standard desktop width: 1000px)
app_container = widgets.VBox(
    [header_layout, menu_container, controls_layout, playlist_output],
    layout=widgets.Layout(
        align_items='center', 
        width='90%',
        max_width='1000px', 
        margin='40px auto',
        padding='50px',
        background_color='#ffffff', 
        border_radius='40px',
        box_shadow='0 20px 80px rgba(0,0,0,0.05)' 
    )
)

# defaualt playlist is deep study
situation_select.value = 'Deep Study'

# click Generate button
generate_playlist_ui(None)

# Render the application
display(app_container)